# OSM Commercial Data Fetcher
**15-minute walk radius · OpenStreetMap via Overpass API**

This notebook downloads all commercial features (shops, offices, amenities, etc.) within a walking-distance radius of any coordinate and exports the result to a timestamped CSV ready for ML analysis.

## 1 · Imports

In [ ]:
import overpy
import requests
import pandas as pd
import math
import time
import glob
from datetime import datetime

print(f"overpy    {overpy.__version__}")
print(f"pandas    {pd.__version__}")
print(f"requests  {requests.__version__}")
print(f"Ready — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2 · OSM Tag Configuration

These constants define **what counts as commercial** in OpenStreetMap.

- `COMMERCIAL_TAGS` — OSM keys that identify commercial features. The order matters: the first matching key becomes the `primary_tag` for each record.
- `AMENITY_EXCLUDE` — amenity values that are infrastructure/civic, not commercial. Extend this list if you see noise in your results.
- `OVERPASS_ENDPOINTS` — mirror servers tried in order. The scraper uses `requests` with a proper `User-Agent` header to avoid being blocked by the API.

> **Only edit this cell if you want to change the scope of what gets collected.**

In [ ]:
# OSM keys that represent commercial land uses (priority order)
COMMERCIAL_TAGS = [
    "shop",
    "amenity",
    "office",
    "craft",
    "tourism",
    "leisure",
]

# Amenity values that are NOT commercial — filtered out before saving
AMENITY_EXCLUDE = {
    "parking", "bench", "waste_basket", "recycling",
    "post_box", "bicycle_parking", "drinking_water",
    "telephone", "toilets", "street_lamp", "fire_station",
    "school", "place_of_worship", "clock",
    "bicycle_rental", "parking_entrance", "vending_machine",
    "fountain", "charging_station", "device_charging_station",
    "bus_station", "social_facility", "community_centre",
    "police", "university", "college", "kindergarten",
}

# Add these two sets alongside AMENITY_EXCLUDE in Cell 2
LEISURE_EXCLUDE = {
    "garden", "park", "pitch", "playground", "outdoor_seating",
    "swimming_pool", "dog_park", "nature_reserve", "picnic_site",
    "shelter", "watering_place", "common",
}

TOURISM_EXCLUDE = {
    "artwork", "viewpoint", "information", "attraction",
    "picnic_site", "camp_site", "caravan_site",
}

SHOP_EXCLUDE = {"vacant", "yes"}
# Overpass API mirrors — tried in order if the primary server fails
# The scraper uses requests + User-Agent to avoid HTTP 406 blocks
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

print(f"Tracking {len(COMMERCIAL_TAGS)} tag types · {len(AMENITY_EXCLUDE)} amenity exclusions")

## 3 · Parameters  ← edit here

Set the origin point and walking radius for this scrape.

| Parameter | Description | Default |
|---|---|---|
| `LAT` / `LON` | Origin coordinate | Times Square, NYC |
| `WALK_MINUTES` | Radius expressed as walking time | 15 min |
| `WALK_SPEED_M_MIN` | Walking speed in meters/minute | 80 m/min ≈ 4.8 km/h |
| `OUTPUT_PATH` | Custom CSV filename, or `None` for auto | `None` (auto) |

> **Tip:** To find coordinates for any location, right-click on [Google Maps](https://maps.google.com) and copy the lat/lon.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────
LAT              = 40.7589    # Origin latitude
LON              = -73.9851   # Origin longitude
WALK_MINUTES     = 15.0       # Walking radius in minutes
WALK_SPEED_M_MIN = 80.0       # Walking speed (meters per minute)
OUTPUT_PATH      = None       # Set to "myfile.csv" to override auto-naming
# ────────────────────────────────────────────────────────

RADIUS_M = WALK_MINUTES * WALK_SPEED_M_MIN

# Auto-generate filename if not specified
if OUTPUT_PATH is None:
    existing     = glob.glob("osm_commercial_*.csv")
    next_index   = len(existing) + 1
    timestamp    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    OUTPUT_PATH  = f"osm_commercial_{next_index:03d}_{timestamp}.csv"

print(f"Origin  : ({LAT}, {LON})")
print(f"Radius  : {RADIUS_M:.0f} m  ({WALK_MINUTES:.0f} min @ {WALK_SPEED_M_MIN:.0f} m/min)")
print(f"Output  : {OUTPUT_PATH}")

## 4 · Core Functions

These cells define the scraping pipeline. **Do not edit unless you are extending the tool.**

- `build_overpass_query` — assembles the Overpass QL query string
- `haversine` — computes straight-line distance in meters between two coordinates
- `_extract_record` — parses a single OSM node/way into a flat dictionary
- `_query_endpoint` — sends one HTTP POST with a `User-Agent` header, parses the response
- `fetch_osm_data` — tries each mirror in order, collects all nodes and ways
- `save_csv` — adds derived ML features (`has_name`, `distance_band`) and writes the CSV

In [ ]:
def build_overpass_query(lat: float, lon: float, radius: float) -> str:
    """Builds Overpass QL query for commercial nodes and ways within radius (meters)."""
    tag_filters = "\n    ".join(
        [f'node["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
        + [f'way["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
    )
    return f"""
[out:json][timeout:60];
(
    {tag_filters}
);
out center tags;
"""


def haversine(lat1, lon1, lat2, lon2) -> float:
    """Straight-line distance in meters between two lat/lon coordinates."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi       = math.radians(lat2 - lat1)
    dlambda    = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def _extract_record(osm_id, osm_type, lat, lon, tags, origin_lat, origin_lon) -> dict | None:
    """Parses a single OSM feature into a flat dict. Returns None if non-commercial."""
    primary_tag = primary_value = None
    for tag in COMMERCIAL_TAGS:
        if tag in tags:
            val = tags[tag]
            if tag == "shop" and val in SHOP_EXCLUDE:
                return None
            if tag == "amenity" and val in AMENITY_EXCLUDE:
                return None
            if tag == "leisure" and val in LEISURE_EXCLUDE:
                return None
            if tag == "tourism" and val in TOURISM_EXCLUDE:
                return None
            primary_tag, primary_value = tag, val
            break
    if not primary_tag:
        return None

    return {
        "osm_id"          : osm_id,
        "osm_type"        : osm_type,
        "lat"             : round(lat, 7),
        "lon"             : round(lon, 7),
        "distance_m"      : round(haversine(origin_lat, origin_lon, lat, lon), 1),
        "primary_tag"     : primary_tag,
        "primary_value"   : primary_value,
        "name"            : tags.get("name", ""),
        "brand"           : tags.get("brand", ""),
        "shop"            : tags.get("shop", ""),
        "amenity"         : tags.get("amenity", ""),
        "office"          : tags.get("office", ""),
        "craft"           : tags.get("craft", ""),
        "tourism"         : tags.get("tourism", ""),
        "leisure"         : tags.get("leisure", ""),
        "cuisine"         : tags.get("cuisine", ""),
        "opening_hours"   : tags.get("opening_hours", ""),
        "phone"           : tags.get("phone", ""),
        "website"         : tags.get("website", ""),
        "wheelchair"      : tags.get("wheelchair", ""),
        "outdoor_seating" : tags.get("outdoor_seating", ""),
        "takeaway"        : tags.get("takeaway", ""),
        "delivery"        : tags.get("delivery", ""),
        "level"           : tags.get("level", ""),
        "addr_street"     : tags.get("addr:street", ""),
        "addr_housenumber": tags.get("addr:housenumber", ""),
        "addr_postcode"   : tags.get("addr:postcode", ""),
    }

## 4.1 · Fetch Functions

In [ ]:
## 4 · Fetch Functions

def _query_endpoint(endpoint: str, query: str) -> overpy.Result:
    """POSTs the Overpass query via requests with a User-Agent header to avoid HTTP 406 blocks."""
    response = requests.post(endpoint, data={"data": query}, headers=HEADERS, timeout=90)
    response.raise_for_status()
    return overpy.Overpass().parse_json(response.text)


def fetch_osm_data(lat: float, lon: float, radius: float) -> list[dict]:
    """Queries Overpass API with automatic failover across mirror servers."""
    query = build_overpass_query(lat, lon, radius)
    print(f"  → Querying Overpass API (radius: {radius:.0f} m)...")

    last_error = None
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            result = _query_endpoint(endpoint, query)
            print(f"  ✓ Success via {endpoint}")
            break
        except Exception as e:
            last_error = e
            print(f"  ✗ {endpoint} — {type(e).__name__}: {e}  (trying next mirror...)")
            time.sleep(3)
    else:
        raise RuntimeError(f"All Overpass API endpoints failed. Last error: {last_error}")

    records = []

    for node in result.nodes:
        rec = _extract_record(
            osm_id=node.id, osm_type="node",
            lat=float(node.lat), lon=float(node.lon),
            tags=node.tags, origin_lat=lat, origin_lon=lon,
        )
        if rec:
            records.append(rec)

    for way in result.ways:
        if hasattr(way, "center_lat") and way.center_lat:
            rec = _extract_record(
                osm_id=way.id, osm_type="way",
                lat=float(way.center_lat), lon=float(way.center_lon),
                tags=way.tags, origin_lat=lat, origin_lon=lon,
            )
            if rec:
                records.append(rec)

    print(f"  → {len(records)} commercial features found")
    return records


print("Fetch functions defined — ready to run.")

## 5 · Run the Scraper

This cell executes the full pipeline using the parameters set in **Cell 3**.  
The Overpass API query may take **10–30 seconds** depending on server load.

In [ ]:
## 5 · Run the Scraper

records = fetch_osm_data(LAT, LON, RADIUS_M)

if not records:
    raise ValueError("No commercial features found in the specified area. Check your coordinates.")

df = pd.DataFrame(records)
df.sort_values("distance_m", inplace=True)

print(f"✓ {len(df)} raw records loaded — not saved yet")

## 6 · Explore Results

Quick inspection of the collected data. Use these cells to validate the output before passing it to your ML pipeline.

In [ ]:
# Preview the first rows
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head(10)

In [ ]:
# Top 15 commercial categories by count
print("Top primary_value categories:")
print(df["primary_value"].value_counts().head(15))

# Distribution by primary tag type (shop / amenity / office / etc.)
print("\nBy primary_tag:")
print(df["primary_tag"].value_counts())

## 7. Labeling 

In [ ]:
## 7 · Label Taxonomy

DROP_VALUES = {
    "outdoor_seating", "pitch", "playground", "shelter",
    "watering_place", "picnic_site", "dog_park", "nature_reserve",
    "common", "artwork", "viewpoint", "information", "attraction",
    "sightseeing", "guide", "loading_dock", "outpost", "post_depot",
    # unmapped — not commercial or unclear
    "diplomatic", "yes", "library", "educational_institution",
    "prep_school", "parish", "rectory", "charity",
    "post_office", "car", "bicycle", "hairdresser_supply",
    "newspaper", "vacuum_cleaner", "video", "sauce",
    "watchmaker", "cleaning", "locksmith", "magic",
    "conference_centre", "fuel", "trade", "training",
    "car_repair", "funeral_directors",
}

LABEL_MAP = {
    # food_drink
    "restaurant": "food_drink",     "fast_food": "food_drink",
    "cafe": "food_drink",           "bar": "food_drink",
    "pub": "food_drink",            "food_court": "food_drink",
    "bakery": "food_drink",         "ice_cream": "food_drink",
    "deli": "food_drink",           "confectionery": "food_drink",
    "chocolate": "food_drink",      "coffee": "food_drink",
    "seafood": "food_drink",        "pastry": "food_drink",
    "tea": "food_drink",            "beverages": "food_drink",
    "health_food": "food_drink",    "brewery": "food_drink",
    "salad": "food_drink",          "butcher": "food_drink",
    "grocery": "food_drink",        "kiosk": "food_drink",
    "greengrocer": "food_drink",

    # retail
    "clothes": "retail",            "gift": "retail",
    "jewelry": "retail",            "shoes": "retail",
    "convenience": "retail",        "supermarket": "retail",
    "department_store": "retail",   "mobile_phone": "retail",
    "newsagent": "retail",          "alcohol": "retail",
    "watches": "retail",            "cosmetics": "retail",
    "toys": "retail",               "electronics": "retail",
    "stationery": "retail",         "bag": "retail",
    "wine": "retail",               "books": "retail",
    "fabric": "retail",             "fashion_accessories": "retail",
    "nutrition_supplements": "retail", "houseware": "retail",
    "furniture": "retail",          "art": "retail",
    "antiques": "retail",           "video_games": "retail",
    "musical_instrument": "retail", "vinyl": "retail",
    "collector": "retail",          "camera": "retail",
    "hifi": "retail",               "hardware": "retail",
    "outdoor": "retail",            "pet": "retail",
    "perfumery": "retail",          "chemist": "retail",
    "variety_store": "retail",      "mall": "retail",
    "tobacco": "retail",            "cannabis": "retail",
    "e-cigarette": "retail",        "florist": "retail",
    "leather": "retail",            "haberdashery": "retail",
    "paint": "retail",              "frame": "retail",
    "bed": "retail",                "military_surplus": "retail",
    "wholesale": "retail",          "erotic": "retail",
    "ticket": "retail",             "marketplace": "retail",
    "craft": "retail",

    # personal_care
    "hairdresser": "personal_care", "beauty": "personal_care",
    "massage": "personal_care",     "tattoo": "personal_care",
    "shoemaker": "personal_care",   "shoe_repair": "personal_care",
    "laundry": "personal_care",     "dry_cleaning": "personal_care",
    "copyshop": "personal_care",    "photo": "personal_care",
    "photographer": "personal_care","sauna": "personal_care",
    "tailor": "personal_care",      "sewing": "personal_care",

    # health
    "pharmacy": "health",           "dentist": "health",
    "clinic": "health",             "doctors": "health",
    "optician": "health",           "veterinary": "health",
    "hospital": "health",           "medical_supply": "health",
    "psychiatrist": "health",       "physician": "health",

    # finance
    "bank": "finance",              "atm": "finance",
    "bureau_de_change": "finance",  "financial_advisor": "finance",
    "financial": "finance",         "money_lender": "finance",
    "money_transfer": "finance",    "pawnbroker": "finance",
    "payment_centre": "finance",

    # hospitality
    "hotel": "hospitality",         "hostel": "hospitality",
    "motel": "hospitality",

    # office_professional
    "company": "office_professional",       "lawyer": "office_professional",
    "accountant": "office_professional",    "it": "office_professional",
    "ngo": "office_professional",           "association": "office_professional",
    "advertising_agency": "office_professional", "consulting": "office_professional",
    "engineer": "office_professional",      "telecommunication": "office_professional",
    "estate_agent": "office_professional",  "travel_agency": "office_professional",
    "architect": "office_professional",     "tax_advisor": "office_professional",
    "marketing": "office_professional",     "property_management": "office_professional",
    "union": "office_professional",         "foundation": "office_professional",
    "coworking": "office_professional",     "coworking_space": "office_professional",

    # leisure_entertainment
    "theatre": "leisure_entertainment",     "cinema": "leisure_entertainment",
    "fitness_centre": "leisure_entertainment", "sports": "leisure_entertainment",
    "sports_centre": "leisure_entertainment", "nightclub": "leisure_entertainment",
    "stripclub": "leisure_entertainment",   "arts_centre": "leisure_entertainment",
    "gallery": "leisure_entertainment",     "museum": "leisure_entertainment",
    "events_venue": "leisure_entertainment","escape_game": "leisure_entertainment",
    "karaoke": "leisure_entertainment",     "karaoke_box": "leisure_entertainment",
    "swimming_pool": "leisure_entertainment", "ice_rink": "leisure_entertainment",
    "miniature_golf": "leisure_entertainment", "dojo": "leisure_entertainment",
    "dance": "leisure_entertainment",       "hackerspace": "leisure_entertainment",
    "social_centre": "leisure_entertainment", "fortune_teller": "leisure_entertainment",
    "car_rental": "leisure_entertainment",  "taxi": "leisure_entertainment",
    "luggage_storage": "leisure_entertainment",
}

# Apply
df_clean = df[~df["primary_value"].isin(DROP_VALUES)].copy()
df_clean["label"] = df_clean["primary_value"].map(LABEL_MAP)
df_clean = df_clean.dropna(subset=["label"]).copy()

print(f"Records after cleaning : {len(df_clean)}")
print("\n── Label distribution ──")
print(df_clean["label"].value_counts())

## 8. Saving csv 

In [ ]:
## 8 · Save Cleaned CSV

cleaned_path = "csv files/raw_umbrella_data.csv"
df_clean.to_csv(cleaned_path, index=False, encoding="utf-8")

print(f"✓ {len(df_clean)} cleaned records saved to '{cleaned_path}'")

In [ ]:
# Column overview — useful when handing off to ML
df.dtypes